# Notebook 03 — Hyperparameter Tuning

Runs `RandomizedSearchCV` on the top-3 tree models, saves tuned
models and the single **best_model.pkl** used by the backend.

In [1]:
import sys, os, time, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.preprocessing import load_preprocessed
from src.model_utils import save_model, evaluate_model
from src.evaluation import plot_confusion_matrix, plot_roc_curve, classification_summary, compare_models_table

In [2]:
X_train, X_test, y_train, y_test, scaler = load_preprocessed('../outputs')

Loaded  X_train=(16296, 14)  X_test=(2474, 14)


In [3]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

configs = {
    'Random_Forest': {
        'model': RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [50, 100, 150],
            'max_depth': [3, 6, 9],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [20, 50, 100]
        }
    },
    'Gradient_Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 150],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.05, 0.1],
            'subsample': [0.8, 0.9, 1.0]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False),
        'params': {
            'n_estimators': [50, 100, 150],
            'max_depth': [3, 6, 9],
            'learning_rate': [0.01, 0.05, 0.1],
            'subsample': [0.7, 0.8, 0.9],
            'colsample_bytree': [0.7, 0.8, 1.0]
        }
    }
}

In [4]:
best_models = {}
tune_log = []

for name, cfg in configs.items():
    print(f'\n{"="*60}\nTuning {name}\n{"="*60}')
    t0 = time.time()
    search = RandomizedSearchCV(
        cfg['model'], cfg['params'],
        n_iter=30, cv=5, scoring='roc_auc',
        random_state=42, n_jobs=-1, verbose=0
    )
    search.fit(X_train, y_train)
    elapsed = time.time() - t0
    print(f'Best CV F1 : {search.best_score_:.4f}')
    print(f'Params     : {search.best_params_}')
    print(f'Time       : {elapsed:.1f}s')
    best_models[name] = search.best_estimator_
    tune_log.append({'Model': name, 'Best_CV_F1': search.best_score_, 'Seconds': round(elapsed, 1)})


Tuning Random_Forest


Best CV F1 : 0.8797
Params     : {'n_estimators': 150, 'min_samples_split': 10, 'min_samples_leaf': 20, 'max_depth': 9}
Time       : 26.2s

Tuning Gradient_Boosting


Best CV F1 : 0.9312
Params     : {'subsample': 0.9, 'n_estimators': 150, 'max_depth': 5, 'learning_rate': 0.1}
Time       : 118.3s

Tuning XGBoost


Best CV F1 : nan
Params     : {'subsample': 0.7, 'n_estimators': 150, 'max_depth': 9, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
Time       : 8.5s


In [5]:
tuned_results = []
for name, mdl in best_models.items():
    m, yp, ypr = evaluate_model(mdl, X_test, y_test, model_name=f'{name}_tuned')
    tuned_results.append(m)
    classification_summary(y_test, yp, model_name=f'{name} (tuned)')

print('\n', compare_models_table(tuned_results).to_string(index=False))

  ✓ Sanity checks passed for Random_Forest_tuned

Classification Report - Random_Forest (tuned)
              precision    recall  f1-score   support

    Rejected       0.90      0.88      0.89      2037
    Approved       0.49      0.52      0.50       437

    accuracy                           0.82      2474
   macro avg       0.69      0.70      0.70      2474
weighted avg       0.82      0.82      0.82      2474

  ✓ Sanity checks passed for Gradient_Boosting_tuned

Classification Report - Gradient_Boosting (tuned)
              precision    recall  f1-score   support

    Rejected       0.88      0.93      0.90      2037
    Approved       0.55      0.43      0.49       437

    accuracy                           0.84      2474
   macro avg       0.72      0.68      0.69      2474
weighted avg       0.83      0.84      0.83      2474


Classification Report - XGBoost (tuned)
              precision    recall  f1-score   support

    Rejected       0.89      0.89      0.89      2

In [6]:
# Save tuned models
for name, mdl in best_models.items():
    save_model(mdl, f'{name}_tuned', model_dir='../models')

# Identify best overall
best_entry = max(tuned_results, key=lambda r: float(r['F1_Score']) if isinstance(r['F1_Score'], (int,float)) else float(r['F1_Score']))
best_key = best_entry['Model'].replace('_tuned', '')
save_model(best_models[best_key], 'best_model', model_dir='../models')

pd.DataFrame(tune_log).to_csv('../outputs/tuning_results.csv', index=False)
print(f'\nBest model: {best_key}')
print('All tuned models saved.')

Saved ../models/Random_Forest_tuned.pkl
Saved ../models/Gradient_Boosting_tuned.pkl
Saved ../models/XGBoost_tuned.pkl
Saved ../models/best_model.pkl

Best model: Random_Forest
All tuned models saved.
